# kwargs-pass-through-recipe — ex2: empty-kwargs case — wrap_forward_fn must record `{}` when caller passes none

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kwargs-pass-through-recipe`. Running the final beacon cell reports progress against the `Backprop: Kwargs pass-through` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Kwargs pass-through` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kwargs-pass-through-recipe`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kwargs-pass-through-recipe"
DD_SUBTOPIC = "Backprop: Kwargs pass-through"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Kwargs pass-through Recipe — quick refresher

The wrapper threads kwargs into **two** places: the forward call AND the Recipe. When the caller passes NO kwargs, the Recipe's `kwargs` must be the empty dict `{}` — NOT the function's default values.

**Worked exemplar.** `t.sum` has default `dim=None, keepdim=False`. `wrapped_sum(x)` (no kwargs) → forward sees defaults, but the Recipe stores `kwargs == {}` because nothing was passed at this call site.

### Exercise 2 — empty-kwargs case — wrap_forward_fn must record `{}` when caller passes none

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply wrap_forward_fn so that when the caller passes NO kwargs the Recipe stores `kwargs == {}` exactly — not the forward fn's default values — while still working when kwargs ARE passed.
> Keywords: kwargs, recipe, empty-kwargs, no-default-injection
> ```

**KCs targeted:** `kwargs-pass-through-recipe`, `recipe-kwargs-faithful-to-call-site`

Implement `wrap_forward_fn(fwd_fn)` so it threads kwargs faithfully:

1. Unbox positional args: `raw = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)`.
2. Forward call: `out_raw = fwd_fn(*raw, **kwargs)` (kwargs may be empty — that's fine, Python accepts `**{}`).
3. Wrap: `out = MiniTensor(out_raw)`.
4. Attach a Recipe:
   - `Recipe.func = fwd_fn`
   - `Recipe.args = raw`
   - `Recipe.kwargs = kwargs` (the literal dict passed to the wrapper, NOT the function's defaults)
   - `Recipe.parents = {0: x}` for a unary call. (For this drill we only test unary ops — kwargs handling is the focus.)

**Key invariant.** If the caller writes `wrapped_sum(x)`, the Recipe's `kwargs` must be `{}` — even though `t.sum(x)` internally uses `dim=None, keepdim=False`. Reverse-pass code keys back fns on the ACTUAL kwargs passed at the forward call site; injecting defaults would break that contract.

The test calls the wrapper TWICE: once with no kwargs, once with `dim=1`, and asserts Recipe.kwargs matches the call site in each case.

In [ ]:
def wrap_forward_fn(fwd_fn):
    """Return a MiniTensor-aware wrapper that records kwargs faithfully."""
    raise NotImplementedError()


def _test_ex2():
    # Wrap t.sum.
    wrapped_sum = wrap_forward_fn(t.sum)

    # --- Case A: no kwargs ---
    x = MiniTensor(t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))
    out = wrapped_sum(x)
    assert isinstance(out, MiniTensor), 'output must be MiniTensor'
    # t.sum(x) with no kwargs sums the entire tensor.
    assert t.allclose(out.array, t.tensor(21.0)), (
        f'no-kwargs forward must use default behavior (full reduction): {out.array}'
    )
    assert out.recipe is not None, 'Recipe was never attached'
    assert out.recipe.func is t.sum, f'Recipe.func wrong: {out.recipe.func}'
    assert out.recipe.kwargs == {}, (
        f'no-kwargs case must store empty dict, got {out.recipe.kwargs!r}. '
        'The Recipe must reflect what the CALLER passed, not the fn defaults.'
    )
    assert 'dim' not in out.recipe.kwargs, (
        'Recipe must NOT inject the fn default dim=None into kwargs'
    )
    assert 'keepdim' not in out.recipe.kwargs, (
        'Recipe must NOT inject the fn default keepdim=False into kwargs'
    )

    # --- Case B: WITH kwargs ---
    out2 = wrapped_sum(x, dim=1)
    assert t.allclose(out2.array, t.tensor([6.0, 15.0])), (
        f'dim=1 forward wrong: {out2.array}'
    )
    assert out2.recipe.kwargs == {'dim': 1}, (
        f'with-kwargs case must store {{"dim": 1}}, got {out2.recipe.kwargs!r}'
    )

    # --- Case C: WITH multiple kwargs ---
    out3 = wrapped_sum(x, dim=1, keepdim=True)
    assert out3.array.shape == (2, 1), f'keepdim shape wrong: {out3.array.shape}'
    assert out3.recipe.kwargs == {'dim': 1, 'keepdim': True}, (
        f'multi-kwarg case wrong: {out3.recipe.kwargs!r}'
    )

    # --- Case D: Recipe.args is the UNBOXED raw torch.Tensor, not the MiniTensor ---
    assert len(out.recipe.args) == 1
    assert isinstance(out.recipe.args[0], t.Tensor), (
        f'Recipe.args[0] must be raw torch.Tensor (unboxed), got {type(out.recipe.args[0]).__name__}'
    )
    assert t.allclose(out.recipe.args[0], x.array)

    # --- Case E: independence — modifying out.recipe.kwargs must NOT affect a later call ---
    out_a = wrapped_sum(x)
    out_b = wrapped_sum(x, dim=0)
    assert out_a.recipe.kwargs == {}, 'Case A snapshot must remain {}'
    assert out_b.recipe.kwargs == {'dim': 0}, 'Case B snapshot must be its own dict'
    out_b.recipe.kwargs['extra'] = 'mutation'
    out_c = wrapped_sum(x)
    assert out_c.recipe.kwargs == {}, (
        f'mutating a prior Recipe.kwargs leaked into a new call: {out_c.recipe.kwargs!r}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_raw = fwd_fn(*raw, **kwargs)
        out = MiniTensor(out_raw)
        parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
        out.recipe = Recipe(func=fwd_fn, args=raw, kwargs=dict(kwargs), parents=parents)
        return out
    return tensor_func
```

**Why `dict(kwargs)` instead of the bare `kwargs` reference.** Python passes `**kwargs` as a fresh dict each call, so reusing the reference is usually safe — BUT later test cases mutate Recipe.kwargs to verify no leakage. Copying with `dict(kwargs)` gives each Recipe its own dict, preventing surprises if downstream code edits it.

**Why the empty-kwargs case matters.** Some back fns dispatch on whether a particular kwarg was passed at all (e.g. `sum_back` differs whether `dim` was specified vs the default of summing all axes). Injecting `dim=None` into the Recipe would conflate "caller passed dim=None explicitly" with "caller passed nothing". Faithful empty-dict preservation keeps that distinction.

**Forward call still uses defaults.** `fwd_fn(*raw, **{})` is identical to `fwd_fn(*raw)` — Python's `**` unpacking does nothing for an empty dict, and the fn's default values activate normally. We get default behavior on the forward AND a faithful empty-dict in the Recipe — no conflict.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()